# 02_run_graphrag_agent.ipynb — query with GraphRAG (Ollama)

Vector search → expand graph → answer with `qwen3`.

In [1]:
# %pip install -q neo4j python-dotenv langchain langchain-community langchain-ollama

import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

try:
    from langchain_ollama import ChatOllama, OllamaEmbeddings
except Exception:
    from langchain_community.chat_models import ChatOllama
    from langchain_community.embeddings import OllamaEmbeddings

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_LLM_MODEL = os.getenv("OLLAMA_LLM_MODEL", "qwen3")
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "all-minilm:l12-v2")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

llm = ChatOllama(model=OLLAMA_LLM_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
embeddings = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)

def run_cypher(q: str, params: dict | None = None):
    with driver.session() as s:
        return list(s.run(q, params or {}))


In [2]:
def vector_search_chunks(query: str, k: int = 6):
    q_emb = embeddings.embed_query(query)
    cypher = """
    CALL db.index.vector.queryNodes('chunk_embedding', $k, $embedding)
    YIELD node, score
    RETURN node.id AS chunk_id, node.doc_id AS doc_id, node.text AS text, node.metadata AS metadata, score
    ORDER BY score DESC
    """
    rows = run_cypher(cypher, {"k": k, "embedding": q_emb})
    return [dict(r) for r in rows]

vector_search_chunks("what is this dataset about?", k=3)


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `metadata` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=80, offset=173>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 173, 'line': 4, 'column': 80}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL db.index.vector.queryNodes('chunk_embedding', $k, $embedding)\n    YIELD node, score\n    RETURN node.id AS chunk_id, node.doc_id AS doc_id, node.text AS text, node.metadata AS metadata, score\n    ORDER BY score DESC\n    "


[{'chunk_id': "Alice's Adventures In Wonderland.pdf#000082",
  'doc_id': "Alice's Adventures In Wonderland.pdf",
  'text': "www.freeclassicebooks.com\n28\n   *    *    *    *    *    *    *\n    *    *    *    *    *    *\n  *    *    *    *    *    *    *\n'Come, my head's free at last!' said Alice in a tone of delight, which changed\ninto alarm in another moment, when she found that her shoulders were\nnowhere to be found: all she could see, when she looked down, was an\nimmense length of neck, which seemed to rise like a stalk out of a sea of\ngreen leaves that lay far below her. 'What CAN all that green stuff be?' said Alice. 'And where HAVE my\nshoulders got to? And oh, my poor hands, how is it I can't see you?' She was\nmoving them about as she spoke, but no result seemed to follow, except a\nlittle shaking among the distant green leaves. As there seemed to be no chance of getting her hands up to her head, she\ntried to get her head down to them, and was delighted to find that he

In [5]:
def graph_neighborhood_from_chunks(chunk_ids: list[str], hops: int = 1, limit_edges: int = 120):
    hops = max(1, min(int(hops), 3))

    cypher = f"""
    MATCH (c:Chunk)-[:MENTIONS]->(e)
    WHERE c.id IN $chunk_ids
    WITH collect(DISTINCT e) AS seed
    UNWIND seed AS s
    MATCH p=(s)-[r*1..{hops}]->(t)
    WHERE ALL(x IN r WHERE type(x) <> 'MENTIONS')   // stay in entity graph
    WITH p LIMIT $limit_edges
    RETURN p
    """
    rows = run_cypher(cypher, {"chunk_ids": chunk_ids, "limit_edges": limit_edges})

    facts, seen = [], set()
    for row in rows:
        p = row["p"]
        nodes = list(p.nodes)
        rels = list(p.relationships)
        for i, rel in enumerate(rels):
            s = nodes[i]
            t = nodes[i+1]
            sname = s.get("name") or s.get("id") or "?"
            tname = t.get("name") or t.get("id") or "?"
            f = f"{sname} -[{rel.type}]-> {tname}"
            if f not in seen:
                seen.add(f)
                facts.append(f)
    return facts


In [6]:
def build_context(chunks: list[dict], facts: list[str], max_chunk_chars: int = 4000):
    chunk_blobs = []
    used = 0
    for c in chunks:
        t = c["text"]
        if used + len(t) > max_chunk_chars:
            t = t[: max(0, max_chunk_chars - used)]
        used += len(t)
        chunk_blobs.append(f"[{c['chunk_id']} score={c['score']:.3f}]\n{t}")
        if used >= max_chunk_chars:
            break

    facts_blob = "\n".join(f"- {f}" for f in facts[:200])
    return "\n\n".join([
        "CHUNKS:\n" + "\n\n".join(chunk_blobs),
        "GRAPH FACTS:\n" + facts_blob if facts_blob else "GRAPH FACTS: (none)",
    ])

QA_SYSTEM = """Answer using only the provided context.
If it's not enough, say what's missing.
Cite chunk ids in parentheses when you use them.
"""

def graphrag_answer(question: str, k: int = 6, hops: int = 1):
    chunks = vector_search_chunks(question, k=k)
    facts = graph_neighborhood_from_chunks([c["chunk_id"] for c in chunks], hops=hops)
    context = build_context(chunks, facts)
    msg = [("system", QA_SYSTEM), ("user", f"Question: {question}\n\nContext:\n{context}")]
    return llm.invoke(msg).content

graphrag_answer("Summarize the main concepts in this dataset.", k=6, hops=2)


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `metadata` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=80, offset=173>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 173, 'line': 4, 'column': 80}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL db.index.vector.queryNodes('chunk_embedding', $k, $embedding)\n    YIELD node, score\n    RETURN node.id AS chunk_id, node.doc_id AS doc_id, node.text AS text, node.metadata AS metadata, score\n    ORDER BY score DESC\n    "


'The dataset contains excerpts from *Alice\'s Adventures in Wonderland* and relational graph facts. Key concepts include:  \n1. **Narrative Elements**: Alice’s surreal journey through Wonderland, interactions with eccentric characters (e.g., the Hatter, March Hare, Dormouse, Mock Turtle, and Gryphon), and absurd scenarios like the tea party and her physical transformation into a creature with a long neck (chunk IDs: #000118, #000082, #000160).  \n2. **Character Dynamics**: The Mock Turtle and Gryphon discuss their education, highlighting themes of absurdity and rivalry (chunk ID: #000160).  \n3. **Graph Facts**: Explicit relationships such as Alice interacting with Baby and being located in the Pool of Tears, the Mock Turtle interacting with the Gryphon and Alice (graph facts).  \n\nMissing: Specific details about the "Baby" character or the "Pool of Tears" setting from the text. Cite chunks as needed.'

In [7]:
while True:
    q = input('Q> ').strip()
    if not q or q.lower() in {'exit','quit'}:
        break
    print('\n' + graphrag_answer(q, k=6, hops=2) + '\n')


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `metadata` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=80, offset=173>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 173, 'line': 4, 'column': 80}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL db.index.vector.queryNodes('chunk_embedding', $k, $embedding)\n    YIELD node, score\n    RETURN node.id AS chunk_id, node.doc_id AS doc_id, node.text AS text, node.metadata AS metadata, score\n    ORDER BY score DESC\n    "



The White Rabbit is a fictional character from *Alice's Adventures in Wonderland* (chunk IDs: [A000052](#000052), [A000053](#000053), [A000023](#000023)). He is a hurried, anxious creature who loses his belongings (a fan and white kid gloves) and mistakes Alice for his housemaid (chunk [A000052](#000052)). Alice helps him retrieve these items by entering his house (chunk [A000053](#000053)), though she later realizes the Rabbit is much smaller than her. The Rabbit’s frantic behavior and interactions with Alice are central to the story’s whimsical plot. 

No real-world identity is provided for the White Rabbit in the context. The missing information would include details about his role beyond the story (e.g., cultural references) or his backstory in the book, which are not present here.

